In [18]:
import os
import glob
import json
import xml.etree.ElementTree as ET
from pprint import pprint

# =========================================================
# XML PATH
# =========================================================
BASE_PATH = "/home/hdo/lidc_project/data/lidc-idri"

xml_files = glob.glob(
    os.path.join(
        BASE_PATH,
        "LIDC-XML-only/tcia-lidc-xml/**/*.xml"
    ),
    recursive=True
)

print("TOTAL XML FILES:", len(xml_files))

# =========================================================
# EXTRACT ONE XML
# =========================================================
def extract_patient_info(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # =====================================================
    # SERIES UID
    # =====================================================
    series_uid = None

    for elem in root.iter():

        tag = elem.tag.split("}")[-1]

        if tag == "SeriesInstanceUid":

            series_uid = elem.text
            break

    if series_uid is None:
        return None, None

    # =====================================================
    # PATIENT DATA
    # =====================================================
    patient_data = {

        "series_uid": series_uid,

        # 실제 결절 단위
        "nodules": {}
    }

    nodule_idx = 0

    # =====================================================
    # READING SESSION
    # =====================================================
    for session in root.iter():

        session_tag = session.tag.split("}")[-1]

        if session_tag != "readingSession":
            continue

        # =================================================
        # NODULE
        # =================================================
        for nodule in session:

            nodule_tag = nodule.tag.split("}")[-1]

            if nodule_tag != "unblindedReadNodule":
                continue

            # -------------------------------------------------
            # Nodule ID
            # -------------------------------------------------
            nodule_id = None

            for child in nodule:

                child_tag = child.tag.split("}")[-1]

                if child_tag == "noduleID":

                    nodule_id = child.text
                    break

            if nodule_id is None:
                continue

            # =================================================
            # 같은 noduleID면 merge
            # =================================================
            if nodule_id not in patient_data["nodules"]:

                patient_data["nodules"][nodule_id] = {

                    # 여러 판독자 malignancy 저장
                    "malignancies": [],

                    # polygon + z 저장
                    "rois": []
                }

            # =================================================
            # MALIGNANCY
            # =================================================
            malignancy = None

            for elem in nodule.iter():

                tag = elem.tag.split("}")[-1]

                if tag == "characteristics":

                    for sub in elem:

                        sub_tag = sub.tag.split("}")[-1]

                        if sub_tag == "malignancy":

                            try:
                                malignancy = int(sub.text)

                            except:
                                malignancy = None

            # 저장
            if malignancy is not None:

                patient_data["nodules"][nodule_id][
                    "malignancies"
                ].append(malignancy)

            # =================================================
            # ROI
            # =================================================
            for roi in nodule.iter():

                roi_tag = roi.tag.split("}")[-1]

                if roi_tag != "roi":
                    continue

                z = None
                polygon = []

                for sub in roi:

                    sub_tag = sub.tag.split("}")[-1]

                    # -----------------------------------------
                    # z position
                    # -----------------------------------------
                    if sub_tag == "imageZposition":

                        try:
                            z = float(sub.text)

                        except:
                            z = None

                    # -----------------------------------------
                    # edgeMap
                    # -----------------------------------------
                    if sub_tag == "edgeMap":

                        x = None
                        y = None

                        for child in sub:

                            child_tag = child.tag.split("}")[-1]

                            if child_tag == "xCoord":

                                try:
                                    x = int(child.text)

                                except:
                                    x = None

                            if child_tag == "yCoord":

                                try:
                                    y = int(child.text)

                                except:
                                    y = None

                        if x is not None and y is not None:

                            polygon.append((x, y))

                # polygon 없는 경우 skip
                if len(polygon) == 0:
                    continue

                # ROI 저장
                patient_data["nodules"][nodule_id][
                    "rois"
                ].append({

                    "z": z,

                    "polygon": polygon
                })

    return series_uid, patient_data

# =========================================================
# DATASET BUILD
# =========================================================
dataset = {}

for xml_path in xml_files:

    try:

        patient_id, patient_data = extract_patient_info(
            xml_path
        )

        if patient_id is None:
            continue

        # 이미 존재하는 환자
        if patient_id in dataset:

            existing_nodules = dataset[
                patient_id
            ]["nodules"]

            new_nodules = patient_data[
                "nodules"
            ]

            # =============================================
            # NODULE MERGE
            # =============================================
            for nodule_id, nodule_data in new_nodules.items():

                # 기존 결절
                if nodule_id in existing_nodules:

                    # malignancy merge
                    existing_nodules[nodule_id][
                        "malignancies"
                    ].extend(
                        nodule_data["malignancies"]
                    )

                    # roi merge
                    existing_nodules[nodule_id][
                        "rois"
                    ].extend(
                        nodule_data["rois"]
                    )

                # 신규 결절
                else:

                    existing_nodules[
                        nodule_id
                    ] = nodule_data

        # 신규 환자
        else:

            dataset[patient_id] = patient_data

    except Exception as e:

        print("\nERROR:", xml_path)
        print(e)

# =========================================================
# SAVE
# =========================================================
SAVE_PATH = os.path.join(
    BASE_PATH,
    "processed_dataset.json"
)

with open(SAVE_PATH, "w") as f:

    json.dump(dataset, f, indent=4)

print("\nDATASET SAVED:")
print(SAVE_PATH)

# =========================================================
# CHECK
# =========================================================
print("\nTOTAL PATIENTS:")
print(len(dataset))

# 첫 환자
first_patient = list(dataset.keys())[0]

print("\nFIRST PATIENT:")
print(first_patient)

# 결절 개수
print("\nNODULE COUNT:")
print(
    len(dataset[first_patient]["nodules"])
)

# 첫 결절
first_nodule = list(
    dataset[first_patient]["nodules"].keys()
)[0]

print("\nFIRST NODULE:")
print(first_nodule)

pprint(
    dataset[first_patient]["nodules"][
        first_nodule
    ]
)

# malignancy 확인
print("\nMALIGNANCIES:")
print(
    dataset[first_patient]["nodules"][
        first_nodule
    ]["malignancies"]
)

# ROI 확인
print("\nFIRST ROI:")
pprint(
    dataset[first_patient]["nodules"][
        first_nodule
    ]["rois"][0]
)

TOTAL XML FILES: 1318

DATASET SAVED:
/home/hdo/lidc_project/data/lidc-idri/processed_dataset.json

TOTAL PATIENTS:
1018

FIRST PATIENT:
1.3.6.1.4.1.14519.5.2.1.6279.6001.670107649586205629860363487713

NODULE COUNT:
5

FIRST NODULE:
6272
{'malignancies': [4],
 'rois': [{'polygon': [(194, 413),
                       (195, 412),
                       (196, 411),
                       (196, 410),
                       (196, 409),
                       (196, 408),
                       (196, 407),
                       (196, 406),
                       (196, 405),
                       (195, 404),
                       (194, 403),
                       (193, 402),
                       (192, 402),
                       (191, 402),
                       (190, 402),
                       (189, 402),
                       (188, 402),
                       (187, 403),
                       (186, 403),
                       (185, 404),
                       (185, 405),
    